In [9]:
!pip install tf-keras

   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ------------------------------ --------- 1.3/1.7 MB 9.2 MB/s eta 0:00:01
   ---------------------------------------- 1.7/1.7 MB 7.6 MB/s  0:00:00


In [27]:
!pip install "keras<3.0.0"

In [ ]:
import cv2
import os
from retinaface import RetinaFace


def extract_15_faces_from_video(video_path, output_folder,
                                frame_skip=10,
                                max_faces=15,
                                img_size=224,
                                margin=20,
                                threshold=0.5):
    """
    Extract maximum 15 face crops from a single video,
    skipping every 10th frame.
    """

    os.makedirs(output_folder, exist_ok=True)

    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        print("❌ Could not open video.")
        return

    frame_count = 0
    saved_count = 0

    while True:
        ret, frame = cap.read()

        if not ret or saved_count >= max_faces:
            break

        # Skip frames (process every 10th frame)
        if frame_count % frame_skip == 0:

            try:
                detections = RetinaFace.detect_faces(frame, threshold=threshold)

                if isinstance(detections, dict):

                    for key in detections:
                        x1, y1, x2, y2 = detections[key]["facial_area"]

                        # Add margin safely
                        x1 = max(0, x1 - margin)
                        y1 = max(0, y1 - margin)
                        x2 = min(frame.shape[1], x2 + margin)
                        y2 = min(frame.shape[0], y2 + margin)

                        face = frame[y1:y2, x1:x2]

                        if face.size == 0:
                            continue

                        face = cv2.resize(face, (img_size, img_size))

                        save_path = os.path.join(
                            output_folder,
                            f"frame_{saved_count:02d}.jpg"
                        )

                        cv2.imwrite(save_path, face)
                        saved_count += 1
                        break   # save only first detected face

            except Exception as e:
                print("Detection error:", e)

        frame_count += 1

    cap.release()

    print("✅ Extraction finished")
    print("Total faces saved:", saved_count)

video_path = "C:\\Users\\RGUKT\\Desktop\\deepfake\\CelebDF_final\\train\\Celeb-real\\00000.mp4"
# output_folder = "single_video_output"
# extract_15_faces_from_video(video_path, output_folder)


✅ Extraction finished
Total faces saved: 15


In [ ]:
import os
import cv2
from tqdm import tqdm
from retinaface import RetinaFace

# =====================================
# CONFIGURATION
# =====================================

input_root = "C:\\Users\\RGUKT\\Desktop\\deepfake\\CelebDF_final"          # original video dataset
output_root = "dataset_faces"   # output cropped faces

splits = ["train", "val"]

for split in splits:

    split_input_path = os.path.join(input_root, split)

    if not os.path.exists(split_input_path):
        print(f"{split} folder not found, skipping...")
        continue

    for class_name in os.listdir(split_input_path):

        class_input_path = os.path.join(split_input_path, class_name)

        if not os.path.isdir(class_input_path):
            continue

        print(f"\n Processing: {split}/{class_name}")

        for video_file in tqdm(os.listdir(class_input_path)):

            if video_file.endswith((".mp4", ".avi", ".mov", ".mkv")):

                video_path = os.path.join(class_input_path, video_file)
                video_name = os.path.splitext(video_file)[0]

                save_folder = os.path.join(
                    output_root,
                    split,
                    class_name,
                    video_name
                )

                extract_15_faces_from_video(video_path, save_folder)

print("\n All train/val/test face extraction completed successfully!")

In [1]:
!pip install ultralytics opencv-python

  Using cached pyyaml-6.0.3-cp313-cp313-win_amd64.whl.metadata (2.4 kB)
  Using cached scipy-1.17.0-cp313-cp313-win_amd64.whl.metadata (60 kB)
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ----------------------------------- ---- 1.0/1.2 MB 6.0 MB/s eta 0:00:01
   ---------------------------------------- 1.2/1.2 MB 5.4 MB/s  0:00:00
   ---------------------------------------- 0.0/810.4 kB ? eta -:--:--
   ---------------------------------------- 810.4/810.4 kB 7.9 MB/s  0:00:00
   ---------------------------------------- 0.0/45.7 MB ? eta -:--:--
   - -------------------------------------- 1.8/45.7 MB 9.1 MB/s eta 0:00:05
   --- ------------------------------------ 3.7/45.7 MB 9.1 MB/s eta 0:00:05
   ---- ----------------------------------- 5.5/45.7 MB 8.9 MB/s eta 0:00:05
   ------ --------------------------------- 7.3/45.7 MB 9.0 MB/s eta 0:00:05
   ------- -------------------------------- 8.9/45.7 MB 8.7 MB/s eta 0:00:05
   --------- -----------------------

In [40]:
import cv2
import os
from ultralytics import YOLO


def extract_faces_yolo(video_path,
                       output_folder,
                       model_path="C:\\Users\\RGUKT\\Desktop\\deepfake\\models\\yolov8n-face.pt",
                       frame_skip=10,
                       max_faces=30,
                       img_size=224):

    os.makedirs(output_folder, exist_ok=True)

    # Load YOLO model
    model = YOLO(model_path)
    # model = YOLO("https://github.com/derronqi/yolov8-face/releases/download/v1.0/yolov8n-face.pt")
    # model = YOLO("yolov5-face.pt")

    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        print("Could not open video.")
        return

    frame_count = 0
    saved_count = 0

    while True:
        ret, frame = cap.read()

        if not ret or saved_count >= max_faces:
            break

        if frame_count % frame_skip == 0:

            # Run YOLO inference
            results = model(frame, verbose=False)

            for r in results:
                boxes = r.boxes

                if boxes is not None:
                    for box in boxes:

                        # Get bounding box
                        x1, y1, x2, y2 = map(int, box.xyxy[0])

                        face = frame[y1:y2, x1:x2]

                        if face.size == 0:
                            continue

                        face = cv2.resize(face, (img_size, img_size))

                        save_path = os.path.join(
                            output_folder,
                            f"frame_{saved_count:02d}.jpg"
                        )

                        cv2.imwrite(save_path, face)
                        saved_count += 1
                        break

                break  # only first detection

        frame_count += 1

    cap.release()

    print("Extraction finished")
    print("Total faces saved:", saved_count)

In [38]:
video_path = "C:\\Users\\RGUKT\\Desktop\\deepfake\\CelebDF_final\\train\\Celeb-real\\00000.mp4"
output_folder = "yolo_faces_output"

extract_faces_yolo(video_path, output_folder)

Extraction finished
Total faces saved: 15


In [ ]:
import os
import cv2
from tqdm import tqdm
from retinaface import RetinaFace

# =====================================
# CONFIGURATION
# =====================================

input_root = "C:\\Users\\RGUKT\\Desktop\\deepfake\\CelebDF_final"          # original video dataset
output_root = "dataset_faces"   # output cropped faces

splits = ["train", "val"]

for split in splits:

    split_input_path = os.path.join(input_root, split)

    if not os.path.exists(split_input_path):
        print(f"{split} folder not found, skipping...")
        continue

    for class_name in os.listdir(split_input_path):

        class_input_path = os.path.join(split_input_path, class_name)

        if not os.path.isdir(class_input_path):
            continue

        print(f"\n Processing: {split}/{class_name}")

        for video_file in tqdm(os.listdir(class_input_path)):

            if video_file.endswith((".mp4", ".avi", ".mov", ".mkv")):

                video_path = os.path.join(class_input_path, video_file)
                video_name = os.path.splitext(video_file)[0]

                save_folder = os.path.join(
                    output_root,
                    split,
                    class_name,
                    video_name
                )

                extract_faces_yolo(video_path, save_folder)

print("\n All train/val/test face extraction completed successfully!")

In [ ]:
import os
import cv2
from tqdm import tqdm
from retinaface import RetinaFace

# =====================================
# CONFIGURATION
# =====================================

input_root = "C:\\Users\\RGUKT\\Desktop\\deepfake\\CelebDF_final"          # original video dataset
output_root = "dataset_faces"   # output cropped faces

splits = ["test"]

for split in splits:

    split_input_path = os.path.join(input_root, split)

    if not os.path.exists(split_input_path):
        print(f"{split} folder not found, skipping...")
        continue

    for class_name in os.listdir(split_input_path):

        class_input_path = os.path.join(split_input_path, class_name)

        if not os.path.isdir(class_input_path):
            continue

        print(f"\n Processing: {split}/{class_name}")

        for video_file in tqdm(os.listdir(class_input_path)):

            if video_file.endswith((".mp4", ".avi", ".mov", ".mkv")):

                video_path = os.path.join(class_input_path, video_file)
                video_name = os.path.splitext(video_file)[0]

                save_folder = os.path.join(
                    output_root,
                    split,
                    class_name,
                    video_name
                )

                extract_faces_yolo(video_path, save_folder)

print("\n All train/val/test face extraction completed successfully!")